# 18q Two checks: own-model weighting and the person's labelled mean (THINGS-EEG2)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 01 and 03b, and the THINGS-EEG2 dataset `cross-subject-brain-encoding-eeg` (09g).

Development analysis, not a prespecified test. A: the own full model scored with feature weights from other people's encoders, from the person's own held-out
training predictions, the same weights on permuted features, and a plain 50 to 250 ms window. B: borrowing and the
map with their mean over the labelled images replaced by the person's mean response over those images.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '18q-decoding', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"ks": [250, 500], "draws": [13, 14, 15], "donor_images": 2000, "select_share": 0.25, "window_s": [0.05, 0.25], "seed": 324}')
PEOPLE = ['sub-01', 'sub-02', 'sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-08', 'sub-09', 'sub-10']
TAG = '18q'

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval import h4b_analysis as ha
from cross_subject_encoding.eval import plugin_map as pm
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

import shutil
from cross_subject_encoding.data.splits import subset
with np.load(C.find_artifact("paper_alexnet_kpca1000.npz")) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
subsets = json.loads(C.find_artifact(C.EEG_K_SUBSETS).read_text())
people = PEOPLE
ATTACHED = te.attached(C.EEG_RAW)  # the 09g dataset (flat names) or the release layout
WORK = C.EEG_RAW if ATTACHED else Path("/kaggle/tmp/things_eeg2")
y_train, bio_test = {}, {}
for s in people:
    if not ATTACHED and te.subject_file(WORK, s, "test") is None:
        te.download_subject(s, WORK)
    tr = te.load_part(te.subject_file(WORK, s, "training"))
    y_train[s], times = tr["data"].mean(1).astype(np.float32), tr["times"]
    bio_test[s] = te.load_part(te.subject_file(WORK, s, "test"))["data"].astype(np.float32)
    del tr
    if not ATTACHED:
        shutil.rmtree(WORK / s, ignore_errors=True)
order = lambda s, d, k: np.asarray(subset(subsets, s, d, k))
print(len(people), 'people loaded |', f'{(time.monotonic() - T0) / 60:.1f} min')

In [ ]:
n_ch, n_t = 17, len(times)
N_TRAIN = len(next(iter(y_train.values())))
win = ha.window_index(times, 100)
t3 = lambda a, n: np.asarray(a, np.float64).reshape(n, n_ch, n_t)
post = np.repeat((times >= -1e-9)[None, :], n_ch, 0).ravel()

def sources(ztr, zte):
    tr = np.empty((len(people), len(ztr), n_ch * n_t), np.float32)
    te_ = np.empty((len(people), len(zte), n_ch * n_t), np.float32)
    q = np.concatenate([ztr, zte])
    for i, s in enumerate(people):
        pred = pl.ols_paper(ztr, y_train[s].reshape(len(ztr), -1), q)
        tr[i], te_[i] = pred[:len(ztr)], pred[len(ztr):]
    return tr, te_

def similarity(pred, meas):
    # centred over the 200 test images, then Pearson over the post-onset features
    p, m = pred[:, post] - pred[:, post].mean(0), meas[:, post] - meas[:, post].mean(0)
    p = (p - p.mean(1, keepdims=True)) / (p.std(1, keepdims=True) + 1e-30)
    m = (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-30)
    return m @ p.T / p.shape[1]  # (measured, candidates)

def calibrate(r, how):
    if how == "none":
        return r
    if how == "csls":  # cross-domain similarity local scaling (Conneau et al. 2018), k = 10, as SATTC's geometric expert
        rm = np.sort(r, 1)[:, -10:].mean(1)
        rp = np.sort(r, 0)[-10:, :].mean(0)
        return 2 * r - rm[:, None] - rp[None, :]
    if how == "dual_z":  # each candidate's similarities standardized over the test responses, then each response's
        z = (r - r.mean(0)) / (r.std(0) + 1e-30)
        return (z - z.mean(1, keepdims=True)) / (z.std(1, keepdims=True) + 1e-30)
    raise ValueError(how)

def test_time_map(pred, meas):
    # label-free, transductive (the 200 test responses as a batch, no pairing used): per 100 ms window, re-mix the
    # prediction's channels so that their covariance over images and time points matches the measured responses'
    from scipy.linalg import sqrtm
    p3, m3 = t3(pred, len(pred)), t3(meas, len(meas))
    out = p3.copy()
    for w in np.unique(win):
        cols = win == w
        xp = (p3[:, :, cols] - p3[:, :, cols].mean(0)).transpose(0, 2, 1).reshape(-1, n_ch)
        xm = (m3[:, :, cols] - m3[:, :, cols].mean(0)).transpose(0, 2, 1).reshape(-1, n_ch)
        cp = np.cov(xp, rowvar=False) + 1e-6 * np.eye(n_ch)
        cm = np.cov(xm, rowvar=False) + 1e-6 * np.eye(n_ch)
        a = np.real(np.linalg.inv(sqrtm(cp)) @ sqrtm(cm))
        mean = p3[:, :, cols].mean(0)
        out[:, :, cols] = np.einsum("nct,cd->ndt", p3[:, :, cols] - mean, a) + mean
    return out.reshape(len(pred), -1)

def encoding_r(pred, meas):
    # per-output correlation across the 200 test images with the repetition average, 100-200 ms, averaged
    sel = np.repeat(((times >= 0.1) & (times < 0.2))[None, :], n_ch, 0).ravel()
    p, m = pred[:, sel] - pred[:, sel].mean(0), meas[:, sel] - meas[:, sel].mean(0)
    return float(np.mean((p * m).sum(0) / np.sqrt((p ** 2).sum(0) * (m ** 2).sum(0) + 1e-30)))

def accuracy(r):
    rank = (r > np.diag(r)[:, None]).sum(1)
    return float((rank < 1).mean() * 100), float((rank < 5).mean() * 100)

def ci(x):
    se = x.std(ddof=1) / np.sqrt(len(x))
    t = stats.t.ppf(0.975, len(x) - 1)
    return {"mean": float(x.mean()), "lower95": float(x.mean() - t * se), "upper95": float(x.mean() + t * se), "n": int(len(x))}

ALPHAS = [10.0 ** e for e in range(-2, 9)]
F = 5
N_TEST = len(z_test)

def parts_for(person, idx, d, ztr, zte, avg_train, avg_test):
    # the map and own-model parts: returns {"map": (out-of-fold, test), "own": (out-of-fold, test)}, each (n, channels, times)
    k = len(idx)
    target = np.asarray(y_train[person][idx], np.float64)
    flat = target.reshape(k, -1)
    a_k = t3(avg_train[idx], k)
    map_test = h4b.apply_map(h4b.fit_map(a_k, target, win, seed=d)[0], t3(avg_test, N_TEST), win)
    own = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, d).fit(ztr[idx], flat)
    own_test = t3(own.predict(zte), N_TEST)
    own_oof = t3(pm.out_of_fold(lambda tr, he: own.refit(ztr[idx[tr]], flat[tr]).predict(ztr[idx[he]]), k, F, d), k)
    map_oof = pm.out_of_fold(lambda tr, he: h4b.apply_map(h4b.fit_map(a_k[tr], target[tr], win, seed=d)[0], a_k[he], win), k, F, d)
    return target, {"map": (map_oof, map_test), "own": (own_oof, own_test)}

def stack(target, chosen):
    # the stack: one weight per part for each channel and 100 ms window, fitted on out-of-fold predictions
    w = pm.stack_fit([oof for oof, _ in chosen], target, win)
    return pm.stack_apply(w, [test for _, test in chosen], win).reshape(N_TEST, -1)

def stack_with_mean(target, chosen, intercept):
    # 18n: stack_fit works on image-centred data, so apply it the same way: centre each part's test
    # predictions on that part's mean over the k labelled images, combine, then add an intercept (channels, times)
    w = pm.stack_fit([oof for oof, _ in chosen], target, win)
    out = pm.stack_apply(w, [test - oof.mean(0) for oof, test in chosen], win) + intercept
    return out.reshape(N_TEST, -1)

def similarity_raw(pred, meas):  # the authors' uncentred identification
    p, m = pred[:, post], meas[:, post]
    p = (p - p.mean(1, keepdims=True)) / (p.std(1, keepdims=True) + 1e-30)
    m = (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-30)
    return m @ p.T / p.shape[1]
def feature_r(pred, resp):
    # per channel-time correlation across images (centred over images); pred, resp (n, features)
    p, m = pred - pred.mean(0), resp - resp.mean(0)
    return (p * m).sum(0) / np.sqrt((p ** 2).sum(0) * (m ** 2).sum(0) + 1e-30)

def to_weights(r, how):
    r = np.where(post, r, 0.0)  # post-onset only, as the identification itself
    if how == "weight":  # soft: each feature counts by its positive prediction accuracy
        return np.clip(r, 0, None)
    if how == "select":  # Kay et al. 2008: keep the best-predicted quarter
        cut = np.quantile(r[post], 1 - FIXED["select_share"])
        return (r >= cut).astype(np.float64)
    raise ValueError(how)

def similarity_w(pred, meas, w):
    # centred identification with features scaled by sqrt(w): the Pearson correlation of the scaled features (not a weighted Pearson correlation)
    cols = w > 0
    sw = np.sqrt(w[cols])
    p, m = pred[:, cols] - pred[:, cols].mean(0), meas[:, cols] - meas[:, cols].mean(0)
    p, m = p * sw, m * sw
    p = (p - p.mean(1, keepdims=True)) / (p.std(1, keepdims=True) + 1e-30)
    m = (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-30)
    return m @ p.T / p.shape[1]

src_train, src_test = sources(z_train, z_test)
D = np.random.default_rng(FIXED["seed"]).choice(len(z_train), FIXED["donor_images"], replace=False)  # as 18o
half = np.random.default_rng(FIXED["seed"] + 1).permutation(len(z_train)) % 2  # two folds of training images
window = np.where(post & np.repeat(((times >= FIXED["window_s"][0] - 1e-9) & (times < FIXED["window_s"][1] - 1e-9))[None, :], n_ch, 0).ravel(), 1.0, 0.0)
rows = []
rng = np.random.default_rng(FIXED["seed"])
for i, person in enumerate(people):
    t = time.monotonic()
    others = [j for j in range(len(people)) if j != i]
    avg_train, avg_test = src_train[others].mean(0), src_test[others].mean(0).astype(np.float64)
    meas = bio_test[person].mean(1).reshape(N_TEST, -1).astype(np.float64)
    y_flat = y_train[person].reshape(len(z_train), -1)
    # A: weights for the person's own full model from four sources
    r_donor = np.mean([feature_r(src_train[np.ix_([o for o in others if o != j], D)].mean(0).astype(np.float64),
                                 y_train[people[j]][D].reshape(len(D), -1).astype(np.float64)) for j in others], 0)
    own_oof = np.empty_like(y_flat, dtype=np.float32)
    for h in (0, 1):  # the person's own encoder, two-fold over their training images
        own_oof[half == h] = pl.ols_paper(z_train[half != h], y_flat[half != h], z_train[half == h])
    r_own = feature_r(own_oof.astype(np.float64), y_flat.astype(np.float64))
    w_donor = to_weights(r_donor, "weight")
    perm = rng.permutation(np.where(post)[0])
    w_perm = np.zeros_like(w_donor)
    w_perm[np.where(post)[0]] = w_donor[perm]  # same weights, wrong features
    W = {"none": np.where(post, 1.0, 0.0), "donor_weight": w_donor, "own_oof_weight": to_weights(r_own, "weight"),
         "window_only": window, "donor_weight_permuted": w_perm}
    own_full = src_test[i].astype(np.float64)
    for how, w in W.items():
        t1, t5 = accuracy(similarity_w(own_full, meas, w))
        rows.append({"person": person, "part": "A", "method": "own_full", "k": "all", "draw": "-", "variant": how, "centred": True, "top1": t1, "top5": t5})
    # B: borrowing and the map with the person's labelled mean as intercept, scored uncentred (and centred)
    for d in FIXED["draws"]:
        for k in FIXED["ks"]:
            idx = order(person, d, k)
            target, p = parts_for(person, idx, d, z_train, z_test, avg_train, avg_test)
            ybar = target.mean(0).reshape(-1)
            map_oof, map_test = p["map"]
            own_oof_k, own_test = p["own"]
            preds = {("avg", "plain"): avg_test,
                     ("avg", "labelled_mean"): avg_test - avg_train[idx].mean(0) + ybar,
                     ("h4b", "plain"): map_test.reshape(N_TEST, -1),
                     ("h4b", "labelled_mean"): map_test.reshape(N_TEST, -1) - map_oof.reshape(k, -1).mean(0) + ybar,
                     ("own_ridge", "plain"): own_test.reshape(N_TEST, -1),
                     ("stack", "labelled_mean"): stack_with_mean(target, [p["map"], p["own"]], target.mean(0))}
            for (method, variant), pred in preds.items():
                for centred, sim in ((True, similarity), (False, similarity_raw)):
                    t1, t5 = accuracy(sim(pred, meas))
                    rows.append({"person": person, "part": "B", "method": method, "k": k, "draw": d, "variant": variant, "centred": centred, "top1": t1, "top5": t5})
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / f"{TAG}_scores.csv", index=False)
    print(person, f"{time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
frame = pd.DataFrame(rows)
def per_person(part, method, k, variant, centred, metric="top1"):
    f = frame[(frame.part == part) & (frame.method == method) & (frame.k.astype(str) == str(k)) & (frame.variant == variant) & (frame.centred == centred)]
    return f.groupby("person")[metric].mean().reindex(people).to_numpy()
summary = {"post_hoc": "development; checks of own-model weighting and the labelled mean", "A": {}, "B": {}}
base = per_person("A", "own_full", "all", "none", True)
summary["A"]["none"] = ci(base)
for how in ("donor_weight", "own_oof_weight", "window_only", "donor_weight_permuted"):
    x = per_person("A", "own_full", "all", how, True)
    summary["A"][how] = {"top1": ci(x), "minus_none": ci(x - base)}
summary["A"]["donor_minus_window"] = ci(per_person("A", "own_full", "all", "donor_weight", True) - per_person("A", "own_full", "all", "window_only", True))
for k in FIXED["ks"]:
    block = {}
    for c in (True, False):
        tag = "centred" if c else "uncentred"
        for method, variant in (("avg", "plain"), ("avg", "labelled_mean"), ("h4b", "plain"), ("h4b", "labelled_mean"), ("own_ridge", "plain"), ("stack", "labelled_mean")):
            block[f"{method}_{variant}_{tag}"] = ci(per_person("B", method, k, variant, c))
    for method in ("avg", "h4b"):
        block[f"{method}_labelled_mean_uncentred_minus_centred_plain"] = ci(per_person("B", method, k, "labelled_mean", False) - per_person("B", method, k, "plain", True))
        block[f"{method}_labelled_mean_minus_plain_uncentred"] = ci(per_person("B", method, k, "labelled_mean", False) - per_person("B", method, k, "plain", False))
    summary["B"][f"k{k}"] = block
k_main = 500 if 500 in FIXED["ks"] else FIXED["ks"][-1]
summary["verdict"] = {
    "A_weighting_holds": summary["A"]["own_oof_weight"]["minus_none"]["lower95"] > 0 and summary["A"]["donor_weight_permuted"]["minus_none"]["upper95"] < summary["A"]["donor_weight"]["minus_none"]["mean"],
    "B_map_uncentred_within_3": summary["B"][f"k{k_main}"]["h4b_labelled_mean_uncentred_minus_centred_plain"]["lower95"] > -3}
(C.METRICS_DIR / f"{TAG}_summary.json").write_text(json.dumps(summary, indent=2))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
a_keys = ["none", "window_only", "donor_weight_permuted", "donor_weight", "own_oof_weight"]
ax[0].bar(range(len(a_keys)), [summary["A"][h]["mean"] if h == "none" else summary["A"][h]["top1"]["mean"] for h in a_keys], color=["#7A7A7A", "#9DB4D9", "#CCCCCC", "#2E5EAA", "#C8102E"])
ax[0].set(xticks=range(len(a_keys)), ylabel="top-1 of 200 (%), centred", title="own full model, weighted identification")
ax[0].set_xticklabels([h.replace("_", " ") for h in a_keys], fontsize=7)
b = summary["B"][f"k{k_main}"]
b_keys = [("avg_plain_uncentred", "borrowing"), ("avg_labelled_mean_uncentred", "borrowing + labelled mean"), ("h4b_plain_uncentred", "map"),
          ("h4b_labelled_mean_uncentred", "map + labelled mean"), ("own_ridge_plain_uncentred", "own ridge"), ("stack_labelled_mean_uncentred", "stack")]
ax[1].barh(range(len(b_keys)), [b[key]["mean"] for key, _ in b_keys], color="#C8102E")
ax[1].barh(range(len(b_keys)), [b[key.replace("labelled_mean_uncentred", "plain_centred").replace("plain_uncentred", "plain_centred")]["mean"] if not key.startswith("stack") else b["stack_labelled_mean_centred"]["mean"] for key, _ in b_keys],
           fill=False, edgecolor="k", lw=0.8)
ax[1].set(yticks=range(len(b_keys)), xlabel=f"top-1 of 200 (%), uncentred (bars) vs centred (outline), k = {k_main}", title="the person's labelled mean")
ax[1].set_yticklabels([l for _, l in b_keys], fontsize=8)
ax[1].invert_yaxis()
fig.tight_layout()
fig.savefig(C.METRICS_DIR / f"{TAG}_figure.png", dpi=150)
print(json.dumps({"A": {h: round(v["mean"] if "mean" in v else v["minus_none"]["mean"], 2) for h, v in summary["A"].items()},
                  "B": {k: {n: round(v["mean"], 1) for n, v in blk.items()} for k, blk in summary["B"].items()}, "verdict": summary["verdict"]}, indent=1))

In [ ]:
export_to_dataset(slug="cross-subject-brain-encoding-18q-outputs", title="XSUB 18q decoding outputs",
                  files=[C.METRICS_DIR / f"18q_{x}" for x in ('summary.json', 'scores.csv', 'figure.png')],
                  include_dirs=[], message="development run")